# 0. Install Dependencies

In [1]:
# !pip install tensorflow==2.3.0
# !pip install gym
# !pip install keras
# !pip install keras-rl2

# 1. Test Random Environment with OpenAI Gym

In [2]:
from gym import Env
from gym.spaces import MultiDiscrete, Box
import numpy as np
import mujoco_py

In [3]:
class SnakeEnv(Env):
    def __init__(self):
        # Define snake model path
        self.snake_model_path = "Model/simplified-snake.xml"
        # Load snake model
        self.snake_model = mujoco_py.load_model_from_path(self.snake_model_path)
        # Load simulation engine
        self.sim = mujoco_py.MjSim(self.snake_model)
        # Visualize simulation environment
        self.viewer = mujoco_py.MjViewer(self.sim)
        # Actions we can take, left, stay, right
        self.action_space = MultiDiscrete([3, 3])
        # Distance array
        self.observation_space = Box(low=np.array([0]), high=np.array([72]))
        # Set goal pos
        self.goal = [6, 2.5]
        # Set start distance
        self.dist = (self.goal[0] - self.sim.data.qpos[6])**2 + (self.goal[1] - self.sim.data.qpos[7])**2
        self.previous_dist = self.dist
        self.steps = 25000

    def step(self, action):
        # Apply action
        self.previous_dist = self.dist
        self.sim.data.ctrl[0] = (action[0]-1) * 0.01
        self.sim.data.ctrl[1] = (action[1]-1) * 0.01
        self.sim.step()
        self.dist = (self.goal[0] - self.sim.data.qpos[6])**2 + (self.goal[1] - self.sim.data.qpos[7])**2
        # Reduce simulation length by 1 step
        self.steps -= 1

        # Calculate reward
        if self.dist < self.previous_dist:
            reward = 1
        else:
            reward = -1

        # Check if simulation is done
        if self.steps <= 0 or self.dist <= 2:
            done = True
        else:
            done = False

        # Set placeholder for info
        info = {}

        # Return step information
        return self.dist, reward, done, info

    def render(self, mode='human'):
        if mode == 'human':
            self.viewer.render()
        if mode == 'rgb_array':
            return self.viewer._read_pixels_as_in_window()

    def reset(self):
        # Reset pos
        # Load simulation engine
        self.sim = mujoco_py.MjSim(self.snake_model)
        # Visualize simulation environment
        self.viewer = mujoco_py.MjViewer(self.sim)
        self.dist = (self.goal[0] - self.sim.data.qpos[6])**2 + (self.goal[1] - self.sim.data.qpos[7])**2
        self.previous_dist = self.dist
        self.steps = 25000

        return self.dist


In [4]:
env = SnakeEnv()

Creating window glfw


/Users/yangzhao/opt/miniconda3/lib/python3.8/site-packages/gym/spaces/box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(


In [5]:
env.observation_space.sample()

array([64.7126], dtype=float32)

In [6]:
env.action_space.sample()

array([0, 2])

In [7]:
# episodes = 2
# for episode in range(1, episodes + 1):
#     dist = env.reset()
#     done = False
#     score = 0
#
#     while not done:
#         actions = env.action_space.sample()
#         n_dist, reward, done, info = env.step(actions)
#         #env.render()
#         score += reward
#     print('Episode:{} Score:{}'.format(episode, score))

# 2. Build Agent

In [8]:
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np

In [9]:
agent_model = PPO('MlpPolicy', env, verbose=1)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
agent_model.learn(total_timesteps=25000)

Creating window glfw


In [ ]:
agent_model.save("Snake_PPO")

In [ ]:
# mean_reward, standard_reward = evaluate_policy(agent_model, agent_model.get_env(), n_eval_episodes=2)

In [ ]:
# print(mean_reward)

In [ ]:
# env.observation_space.shape

In [ ]:
# obs = np.array([env.reset()])

In [ ]:
# obs.shape

In [ ]:
# # del agent_model
# import cv2
# import matplotlib.pyplot as plt
#
#
# from gym.wrappers import Monitor
# agent_model = PPO.load("Snake_PPO", env=env)
# # env = Monitor(env, "recording", video_callable=lambda episode_id: True, force=True)
# obs = np.array([env.reset()])
# done = False
# img_array = []
# score = 0
#
# while not done:
#     action, _states = agent_model.predict(obs, deterministic=True)
#     obs, rewards, done, info = env.step(action)
#     obs = np.array([obs])
#     env.render(mode='human')
#     # img = env.render(mode='rgb_array')
#
#     # img_array.append(img)
#     score += rewards
# env.close()
#
#
#
# # height, width, layers = img.shape
# # size = (width,height)
# # print(size)
# # out = cv2.VideoWriter('project.mp4', cv2.VideoWriter_fourcc(*'MP4V'), 30, size)
# # for i in range(len(img_array)):
# #     out.write(img_array[i])
# # out.release()
# print(score)
# print(img_array.shape)